<a href="https://colab.research.google.com/github/warrendevhub/segmentation-rfm-ecommerce/blob/main/Segmentation_RFM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Adresse du jeu de données "Online Retail" (UCI)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx"

# Lecture du fichier Excel dans un tableau (DataFrame)
df = pd.read_excel(url)

# Affiche les 5 premières lignes
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [ ]:
# Nombre de lignes et de colonnes
print(df.shape)

# Type de chaque colonne et valeurs manquantes
df.info()

(541909, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


## 1. Chargement et découverte des données

In [ ]:
# Nombre de valeurs manquantes par colonne
print(df.isna().sum())

# Statistiques des colonnes chiffrées (minimum, maximum, moyenne...)
df.describe()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386048,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


## 2. Nettoyage des données

In [ ]:
# 1. Garder uniquement les lignes qui ont un identifiant client
df_clean = df.dropna(subset=["CustomerID"])

# 2. Retirer les annulations (numéro de facture commençant par "C")
df_clean = df_clean[~df_clean["InvoiceNo"].astype(str).str.startswith("C")]

# 3. Garder uniquement les quantités et les prix positifs
df_clean = df_clean[(df_clean["Quantity"] > 0) & (df_clean["UnitPrice"] > 0)].copy()

# 4. Calculer le montant de chaque ligne (quantité x prix)
df_clean["Montant"] = df_clean["Quantity"] * df_clean["UnitPrice"]

# Bilan du nettoyage
print("Lignes avant :", len(df))
print("Lignes après :", len(df_clean))
print("Clients uniques :", df_clean["CustomerID"].nunique())

Lignes avant : 541909
Lignes après : 397884
Clients uniques : 4338


## 3. Calcul des indicateurs RFM (SQL)

In [ ]:
import duckdb

requete = """
SELECT
    CAST(CustomerID AS INTEGER) AS client,
    DATE_DIFF('day', CAST(MAX(InvoiceDate) AS DATE), DATE '2011-12-10') AS recence,
    COUNT(DISTINCT InvoiceNo) AS frequence,
    ROUND(SUM(Montant), 2) AS montant
FROM df_clean
GROUP BY CustomerID
ORDER BY montant DESC
"""

rfm = duckdb.sql(requete).df()
print("Nombre de clients :", len(rfm))
rfm.head(10)

Nombre de clients : 4338


,client,recence,frequence,montant
0,14646,2,73,280206.02
1,18102,1,60,259657.30
2,17450,9,46,194550.79
3,16446,1,2,168472.50
4,14911,2,201,143825.06
5,12415,25,21,124914.53
6,14156,10,55,117379.63
7,17511,3,31,91062.38
8,16029,39,63,81024.84
9,12346,326,1,77183.60


In [ ]:
# Toutes les lignes brutes de ces deux clients suspects
df[df["CustomerID"].isin([12346, 16446])]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
61619,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom
61624,C541433,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,2011-01-18 10:17:00,1.04,12346.0,United Kingdom
194354,553573,22980,PANTRY SCRUBBING BRUSH,1,2011-05-18 09:52:00,1.65,16446.0,United Kingdom
194355,553573,22982,PANTRY PASTRY BRUSH,1,2011-05-18 09:52:00,1.25,16446.0,United Kingdom
540421,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
540422,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom


### Correction : achats annulés

In [ ]:
# Données avec client identifié et prix positif, en gardant cette fois les annulations
df_net = df.dropna(subset=["CustomerID"]).copy()
df_net = df_net[df_net["UnitPrice"] > 0]
df_net["InvoiceNo"] = df_net["InvoiceNo"].astype(str)
df_net["StockCode"] = df_net["StockCode"].astype(str)
df_net["Montant"] = df_net["Quantity"] * df_net["UnitPrice"]

requete = """
WITH achats AS (
    SELECT * FROM df_net
    WHERE InvoiceNo NOT LIKE 'C%' AND Quantity > 0
),
annulations AS (
    SELECT * FROM df_net
    WHERE InvoiceNo LIKE 'C%'
),
achats_valides AS (
    SELECT a.*
    FROM achats a
    WHERE NOT EXISTS (
        SELECT 1 FROM annulations c
        WHERE c.CustomerID = a.CustomerID
          AND c.StockCode = a.StockCode
          AND c.Quantity = -a.Quantity
          AND c.InvoiceDate >= a.InvoiceDate
    )
)
SELECT
    CAST(CustomerID AS INTEGER) AS client,
    DATE_DIFF('day', CAST(MAX(InvoiceDate) AS DATE), DATE '2011-12-10') AS recence,
    COUNT(DISTINCT InvoiceNo) AS frequence,
    ROUND(SUM(Montant), 2) AS montant
FROM achats_valides
GROUP BY CustomerID
ORDER BY montant DESC
"""

rfm = duckdb.sql(requete).df()
print("Nombre de clients :", len(rfm))

# Vérification sur nos deux clients suspects
print(rfm[rfm["client"].isin([12346, 16446])])

rfm.head(10)

Nombre de clients : 4327
      client  recence  frequence  montant
4326   16446      206          1      2.9


,client,recence,frequence,montant
0,14646,2,72,278528.42
1,18102,1,60,259657.30
2,17450,9,45,189767.53
3,14911,2,197,131839.01
4,12415,25,19,123806.88
5,14156,10,55,114395.83
6,17511,3,30,79114.38
7,16684,5,28,65920.12
8,14096,5,17,65164.79
9,13694,4,50,63236.74


In [ ]:
# Scores de 1 à 5 par quintiles
rfm["R"] = pd.qcut(rfm["recence"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F"] = pd.qcut(rfm["frequence"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M"] = pd.qcut(rfm["montant"], 5, labels=[1, 2, 3, 4, 5]).astype(int)

# Attribution d'un segment selon la récence (R) et la fréquence (F)
def segment(ligne):
    R, F = ligne["R"], ligne["F"]
    if R >= 4 and F >= 4:
        return "Champions"
    elif R == 3 and F >= 4:
        return "Clients fidèles"
    elif R <= 2 and F >= 3:
        return "À risque"
    elif R >= 4:
        return "Nouveaux / prometteurs"
    elif R == 3:
        return "À relancer"
    else:
        return "Perdus"

rfm["segment"] = rfm.apply(segment, axis=1)
rfm.head(10)

,client,recence,frequence,montant,R,F,M,segment
0,14646,2,72,278528.42,5,5,5,Champions
1,18102,1,60,259657.30,5,5,5,Champions
2,17450,9,45,189767.53,5,5,5,Champions
3,14911,2,197,131839.01,5,5,5,Champions
4,12415,25,19,123806.88,4,5,5,Champions
5,14156,10,55,114395.83,5,5,5,Champions
6,17511,3,30,79114.38,5,5,5,Champions
7,16684,5,28,65920.12,5,5,5,Champions
8,14096,5,17,65164.79,5,5,5,Champions
9,13694,4,50,63236.74,5,5,5,Champions


In [ ]:
# Tableau récapitulatif par segment
resume = rfm.groupby("segment").agg(
    nb_clients=("client", "count"),
    recence_moy=("recence", "mean"),
    frequence_moy=("frequence", "mean"),
    montant_moy=("montant", "mean"),
    ca_total=("montant", "sum"),
).round(1)

resume["part_clients_%"] = (resume["nb_clients"] / resume["nb_clients"].sum() * 100).round(1)
resume["part_ca_%"] = (resume["ca_total"] / resume["ca_total"].sum() * 100).round(1)

resume.sort_values("ca_total", ascending=False)

,nb_clients,recence_moy,frequence_moy,montant_moy,ca_total,part_clients_%,part_ca_%
segment,,,,,,,
Champions,1108,13.4,10.0,5124.1,5677462.7,25.6,67.5
Clients fidèles,321,50.1,5.7,2292.0,735719.2,7.4,8.7
À risque,653,153.0,3.4,1015.9,663402.2,15.1,7.9
Perdus,1074,217.3,1.1,493.7,530281.0,24.8,6.3
Nouveaux / prometteurs,632,17.9,1.8,708.6,447813.3,14.6,5.3
À relancer,539,52.8,1.6,660.1,355811.0,12.5,4.2


## 5. Conclusions et recommandations

**Constat principal :** 25,6 % des clients (les Champions) génèrent 67,5 % du chiffre d'affaires. Avec les clients fidèles, un tiers des clients réalise plus des trois quarts du chiffre d'affaires (76,2 %). La boutique dépend donc fortement d'un petit groupe de clients.

**Recommandations par segment :**
- **Champions (1 108 clients, 67,5 % du CA)** : priorité absolue à la fidélisation (programme VIP, avant-premières, service prioritaire). Perdre 10 % de ces clients coûterait près de 7 % du chiffre d'affaires.
- **À risque (653 clients, 7,9 % du CA)** : ces clients commandaient en moyenne 3,4 fois mais n'ont rien acheté depuis 153 jours. Contrairement aux clients perdus, ils connaissent la boutique et restent récupérables : campagne de reconquête rapide (offre personnalisée, message « vous nous manquez »).
- **Nouveaux / prometteurs (632 clients, 5,3 % du CA)** : clients récents (18 jours) mais encore peu engagés (1,8 commande). Objectif : déclencher une 2e puis une 3e commande pour en faire de futurs Champions.
- **Clients fidèles et à relancer** : relances régulières sur les nouveautés pour éviter qu'ils ne basculent dans le segment « à risque ».
- **Perdus (1 074 clients, 6,3 % du CA)** : une seule commande en moyenne et plus de 7 mois sans achat. Investissement minimal : quelques emails automatiques à faible coût.

**Limites de l'analyse :**
- Seules les annulations exactes (même client, même produit, même quantité) ont été retirées ; les annulations partielles restent comptées.
- Les segments reposent sur la récence et la fréquence ; le montant sert à mesurer le poids de chaque segment.
- Les données couvrent une seule année (décembre 2010 à décembre 2011) d'une boutique en ligne principalement britannique.

In [ ]:
from google.colab import files

# Export du tableau RFM (un client par ligne) en fichier CSV
rfm.to_csv("rfm_clients.csv", index=False)
files.download("rfm_clients.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>